# Stochastic Calculus, Step by Step · Lesson 13: Brownian motion

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. From a random walk to Brownian motion · 3. The definition, checked on simulated paths · 4. Covariance: min(s, t) · 5. Zooming in: self-similarity · 6. Three martingales · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 13,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. From a random walk to Brownian motion

Take $n$ coin tosses per unit of time, each moving a walker $\pm 1/\sqrt n$. After time $t$ the walker has taken $nt$ steps, so its position has mean 0 and variance $nt \times (1/\sqrt n)^2 = t$, **whatever $n$ is**. As $n$ grows, the central limit theorem (Lesson 5) makes the position at time 1 normal: the limit is Brownian motion.

In [ ]:
rng = np.random.default_rng(seed)
walk_rows, walk_cdfs = [], {}
xs = np.linspace(-3, 3, 601)
for n in (10, 100, 1000, 10_000):
    heads = rng.binomial(n, 0.5, 400_000)              # number of +1 steps out of n
    W1 = (2 * heads - n) / math.sqrt(n)                # position at time 1
    walk_rows.append({"n": n, "mean": float(W1.mean()), "var": float(W1.var()),
                      "p_below_1": float((W1 <= 1).mean()), "n_values": int(n + 1)})
    walk_cdfs[str(n)] = (np.searchsorted(np.sort(W1), xs, side="right") / W1.size).tolist()   # P(W1 <= x)
out["walk"] = {"rows": walk_rows, "N1": N(1.0),
               "fig": {"x": xs.tolist(), "cdfs": walk_cdfs, "normal": [N(x) for x in xs], "x_range": [-3, 3],
                       "ticks": [{"x": x, "label": f"{x:g}"} for x in (-3, -2, -1, 0, 1, 2, 3)]}}
# a few walk paths on [0, 1] for the chart: coarse and fine
walk_paths = {}
for n in (10, 100, 10_000):
    steps_ = rng.choice([-1.0, 1.0], (8, n)) / math.sqrt(n)
    every = max(1, n // 500)
    walk_paths[str(n)] = {"t": np.linspace(0, 1, n + 1)[::every].tolist(),
                          "paths": np.concatenate([np.zeros((8, 1)), np.cumsum(steps_, axis=1)], axis=1)[:, ::every].round(4).tolist()}
out["walk"]["paths"] = walk_paths
pd.DataFrame(walk_rows)

## 3. The definition, checked on simulated paths

**Brownian motion** $W_t$ is a random path with (1) $W_0 = 0$; (2) **independent increments**: the moves over separate time intervals are independent; (3) **normal increments**: $W_t - W_s \sim N(0, t - s)$ for $s < t$; (4) **continuous paths**. Simulate it by adding independent $N(0, \Delta t)$ steps.

In [ ]:
n_paths, n_steps = 20_000, 1000
dt = 1.0 / n_steps
dW = rng.normal(0.0, math.sqrt(dt), (n_paths, n_steps))
W = np.concatenate([np.zeros((n_paths, 1)), np.cumsum(dW, axis=1)], axis=1)
grid = np.linspace(0, 1, n_steps + 1)
at = lambda t: W[:, int(round(t * n_steps))]
defn = {"rows": [{"t": t, "mean": float(at(t).mean()), "var": float(at(t).var())} for t in (0.25, 0.5, 1.0)],
        "incr_corr": float(np.corrcoef(at(0.5) - at(0.25), at(1.0) - at(0.5))[0, 1]),
        "incr_var": float((at(1.0) - at(0.5)).var())}
out["defn"] = defn
out["fan"] = {"t": grid[::10].tolist(), "paths": W[:12, ::10].round(4).tolist(),
              "band": np.sqrt(grid[::10]).tolist(), "band_lo": (-np.sqrt(grid[::10])).tolist(),
              "ticks": [{"x": x, "label": f"{x:g}"} for x in (0, 0.25, 0.5, 0.75, 1)]}
defn

## 4. Covariance: min(s, t)

For $s < t$ write $W_t = W_s + (W_t - W_s)$. The second piece is independent of $W_s$ and has mean 0, so $\mathrm{Cov}(W_s, W_t) = \mathrm{Var}(W_s) = s$. In general $\mathrm{Cov}(W_s, W_t) = \min(s, t)$.

In [ ]:
cov_rows = []
for s, t in ((0.25, 0.75), (0.5, 1.0), (0.3, 0.6), (0.8, 0.9)):
    cov_rows.append({"s": s, "t": t, "sim": float(np.mean(at(s) * at(t))), "theory": min(s, t),
                     "corr": float(np.corrcoef(at(s), at(t))[0, 1]), "corr_theory": math.sqrt(s / t)})
out["cov"] = {"rows": cov_rows, "var_sum_12": 1 + 2 + 2 * 1}
pd.DataFrame(cov_rows)

## 5. Zooming in: self-similarity

Stretch time by $c$ and shrink space by $\sqrt c$: $\frac{1}{\sqrt c}W_{ct}$ is again a Brownian motion. So a Brownian path looks equally rough at every scale. One fine path, zoomed into $[0, 1/16]$ and $[0, 1/256]$ and rescaled:

In [ ]:
fine_n = 2**16
fine = np.concatenate([[0], np.cumsum(rng.normal(0, math.sqrt(1 / fine_n), fine_n))])
zoom = {}
for level, c in (("1", 1), ("16", 16), ("256", 256)):
    m = fine_n // c                                     # points in [0, 1/c]
    idx = np.linspace(0, m, 513).astype(int)
    zoom[level] = {"t": (idx / m).tolist(), "w": (fine[idx] * math.sqrt(c)).tolist(),
                   "range": float(np.ptp(fine[: m + 1]) * math.sqrt(c))}
out["zoom"] = {"levels": zoom, "ticks": [{"x": x, "label": f"{x:g}"} for x in (0, 0.25, 0.5, 0.75, 1)]}
{k_: round(v["range"], 3) for k_, v in zoom.items()}

## 6. Three martingales

With $\mathcal F_s$ the information in the path up to time $s$, three processes are martingales (Lesson 11): $W_t$, $\;W_t^2 - t$, and the **exponential martingale** $Z_t = e^{\sigma W_t - \frac12\sigma^2t}$. Each proof splits $W_t = W_s + (W_t - W_s)$ and uses independence. The last one gives the share: with $S_t = S_0e^{(r - \frac12\sigma^2)t + \sigma W_t}$, the discounted price $e^{-rt}S_t = S_0Z_t$ is a martingale.

Check: their averages stay at their starting values, and, sorting paths by $W_{0.5}$, the average of the time-1 value in each group matches the time-0.5 value.

In [ ]:
Z = lambda w, t: np.exp(sigma * w - 0.5 * sigma**2 * t)
mart = {"E_W1": float(at(1.0).mean()), "E_W1sq_minus_1": float((at(1.0) ** 2 - 1).mean()),
        "E_Z1": float(Z(at(1.0), 1.0).mean()), "E_Z1_theory": 1.0,
        "E_disc_S": float((S0 * Z(at(1.0), 1.0)).mean())}
# conditional check: group paths by W_0.5
bins = np.quantile(at(0.5), np.linspace(0, 1, 11))
g = np.clip(np.digitize(at(0.5), bins[1:-1]), 0, 9)
rows = []
for k_ in range(10):
    sel = g == k_
    rows.append({"group": k_ + 1, "W_half": float(at(0.5)[sel].mean()),
                 "sq_now": float((at(0.5)[sel] ** 2 - 0.5).mean()), "sq_later": float((at(1.0)[sel] ** 2 - 1).mean()),
                 "Z_now": float(Z(at(0.5)[sel], 0.5).mean()), "Z_later": float(Z(at(1.0)[sel], 1.0).mean())})
mart["groups"] = rows
mart["max_gap_sq"] = max(abs(r_["sq_now"] - r_["sq_later"]) for r_ in rows)
mart["max_gap_Z"] = max(abs(r_["Z_now"] - r_["Z_later"]) for r_ in rows)
out["mart"] = mart
pd.DataFrame(rows).round(4)

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 3: Solutions*.

**Exercise 1.** A walker takes $n$ steps per unit of time, each $+1/\sqrt n$ or $-1/\sqrt n$ with probability $\tfrac12$. Show that at time $t$ (with $nt$ a whole number) its position has mean 0 and variance $t$. Why do the steps have to be of size $1/\sqrt n$, and not $1/n$?

**Exercise 2.** Prove that $\mathrm{Cov}(W_s, W_t) = \min(s, t)$. Use it to find $\mathrm{Var}(W_1 + W_2)$ and the correlation of $W_1$ and $W_4$.

**Exercise 3.** Find $P(W_1 > 0 \text{ and } W_2 > W_1)$. You are told that $P(W_1 > 0 \text{ and } W_2 > 0) = \tfrac38$. Why are the two different?

**Exercise 4.** Let $c > 0$ and $B_t = W_{ct}/\sqrt c$. Check that $B$ has the four properties of a Brownian motion.

**Exercise 5.** Prove that $W_t^2 - t$ is a martingale. (Write $W_t = W_s + (W_t - W_s)$ and expand the square.)

**Exercise 6.** Prove that $Z_t = e^{\sigma W_t - \frac12\sigma^2t}$ is a martingale, using $E[e^{uX}] = e^{u^2v/2}$ for $X \sim N(0, v)$ (Lesson 4). Deduce that $e^{-rt}S_t$ is a martingale when $S_t = S_0e^{(r - \frac12\sigma^2)t + \sigma W_t}$.

**Exercise 7.** (Code) Simulate scaled random walks with $n = 10$, $100$ and $10{,}000$ steps per unit of time, 200,000 times each, and compare $P(W^{(n)}_1 \le 1)$ with $N(1)$. Why is the answer for small $n$ so far off?

**Exercise 8.** (Code) Simulate 50,000 Brownian paths on $[0, 2]$ and check $\mathrm{Cov}(W_s, W_t) = \min(s, t)$ on a grid of pairs, and $E[Z_t] = 1$ for $t = 0.5, 1, 2$ with $\sigma = 0.2$.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")